# LeetCode 1456 — Maximum Number of Vowels in a Substring of Given Length bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-vowels-in-a-substring-of-given-length/ · Độ khó: **Medium** · Chủ đề: String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maxVowels(self, s: str, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận dạng bài **cửa sổ trượt kích thước cố định** (fixed-size sliding window): xét mọi đoạn con liên tiếp dài đúng `k`.
2. Viết brute force `O(n·k)` và hiểu vì sao phần lớn công việc bị **lặp lại** giữa hai cửa sổ kề nhau.
3. Dùng **mảng cộng dồn (prefix sum)** để đếm nguyên âm trong một đoạn bất kỳ trong `O(1)`.
4. Dùng **cửa sổ trượt `O(n)`, bộ nhớ `O(1)`**: thêm ký tự vào bên phải, bỏ ký tự ra bên trái.
5. Tránh lỗi: thiếu cửa sổ cuối, quên bỏ ký tự cũ, bỏ nhầm chỉ số, cập nhật đáp án khi cửa sổ đang dài `k+1`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`. Trả về **số nguyên âm nhiều nhất** có trong một **chuỗi con độ dài đúng `k`** của `s`. Nguyên âm tiếng Anh: `'a', 'e', 'i', 'o', 'u'`.

**Ví dụ 1:** `s = "abciiidef", k = 3` → `3` (chuỗi con `"iii"`).

**Ví dụ 2:** `s = "aeiou", k = 2` → `2` (mọi chuỗi con độ dài 2 đều có 2 nguyên âm).

**Ví dụ 3:** `s = "leetcode", k = 3` → `2` (`"lee"`, `"eet"`, `"ode"` đều có 2).

**Ràng buộc:** `1 <= s.length <= 10^5`, `s` chỉ gồm chữ thường tiếng Anh, `1 <= k <= s.length`.

In [1]:
import random
import string
import time
from itertools import accumulate
from typing import List

examples = [(("abciiidef", 3), 3), (("aeiou", 2), 2), (("leetcode", 3), 2)]
for (s, k), want in examples:
    print(f"s={s!r}, k={k} -> kỳ vọng {want}")

s='abciiidef', k=3 -> kỳ vọng 3
s='aeiou', k=2 -> kỳ vọng 2
s='leetcode', k=3 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Chuỗi con (substring) và slicing

Chuỗi con độ dài `k` bắt đầu tại `i` là `s[i:i+k]`. Chỉ số bắt đầu hợp lệ: `i = 0, 1, …, n−k` ⇒ có **`n − k + 1`** chuỗi con, tức `range(n - k + 1)`.

In [2]:
s, k = "leetcode", 3
subs = [s[i:i + k] for i in range(len(s) - k + 1)]
print(subs, "| số lượng =", len(subs), "= n-k+1 =", len(s) - k + 1)

['lee', 'eet', 'etc', 'tco', 'cod', 'ode'] | số lượng = 6 = n-k+1 = 6


### 3.2 Kiểm tra nguyên âm: `in` với `set` / chuỗi

`c in "aeiou"` hoặc `c in {"a","e","i","o","u"}` đều `O(1)` vì tập rất nhỏ. Đếm nguyên âm của một chuỗi: `sum(c in VOWELS for c in t)` (True = 1, False = 0).

In [3]:
VOWELS = set("aeiou")
print([c in VOWELS for c in "lee"])
print(sum(c in VOWELS for c in "leetcode"), "nguyên âm trong 'leetcode'")
print(True + True + False)

[False, True, True]
4 nguyên âm trong 'leetcode'
2


### 3.3 Mảng cộng dồn (prefix sum) với `itertools.accumulate`

Đặt `isv[i] = 1` nếu `s[i]` là nguyên âm. `P[0] = 0`, `P[i+1] = P[i] + isv[i]`. Số nguyên âm trong `s[i:j]` là `P[j] − P[i]`.

In [4]:
s = "leetcode"
isv = [1 if c in VOWELS else 0 for c in s]
P = [0] + list(accumulate(isv))
print("isv =", isv)
print("P   =", P)
print("nguyên âm trong s[1:4] =", P[4] - P[1], "| trực tiếp:", sum(isv[1:4]), repr(s[1:4]))

isv = [0, 1, 1, 0, 0, 1, 0, 1]
P   = [0, 0, 1, 2, 2, 2, 3, 3, 4]
nguyên âm trong s[1:4] = 2 | trực tiếp: 2 'eet'


### 3.4 `max` với generator

`max(f(i) for i in range(...))` lấy giá trị lớn nhất mà không cần tạo list trung gian.

In [5]:
print(max(P[i + 3] - P[i] for i in range(len(s) - 3 + 1)))

2


### 3.5 Phép toán trên `bool` trong Python

`bool` là lớp con của `int`: `True == 1`, `False == 0`. Nhờ vậy ta viết gọn `cnt += (s[i] in VOWELS) - (s[i-k] in VOWELS)` — hiệu hai giá trị bool cho `-1`, `0` hoặc `1`. **Ngoặc là bắt buộc**: toán tử `in` có độ ưu tiên **thấp hơn** `-`, nên viết thiếu ngoặc sẽ bị hiểu thành một biểu thức hoàn toàn khác (ở đây là phép trừ tập hợp với số ⇒ lỗi).

In [6]:
print(isinstance(True, int), True - False, False - True, True - True)
x, y = "a", "b"
print((x in VOWELS) - (y in VOWELS), "<- có ngoặc: đúng ý")
try:
    print(x in VOWELS - (y in VOWELS))
except TypeError as e:
    print("thiếu ngoặc -> TypeError:", e)

True 1 -1 0
1 <- có ngoặc: đúng ý
thiếu ngoặc -> TypeError: unsupported operand type(s) for -: 'set' and 'bool'


### 3.6 Vì sao cửa sổ trượt "vào/ra" áp dụng được?

Cửa sổ trượt cố định dùng được khi đại lượng cần theo dõi **cập nhật được khi thêm 1 phần tử và bỏ 1 phần tử** trong `O(1)`: tổng, số đếm, tích (không có số 0), tần suất từng ký tự… Với đại lượng như **max/min** thì "bỏ ra" không dễ — cần deque đơn điệu (xem LeetCode 239, 1438). Bài này là số đếm nên rất thuận lợi.

In [7]:
# Minh hoạ: số đếm cập nhật O(1); max thì không (bỏ phần tử lớn nhất ra phải tìm lại max)
w = [3, 9, 2]
print("cửa sổ", w, "tổng", sum(w), "max", max(w))
w2 = w[1:] + [1]
print("trượt ->", w2, "tổng mới = tổng cũ - 3 + 1 =", sum(w) - 3 + 1, "| max mới phải tính lại:", max(w2))

cửa sổ [3, 9, 2] tổng 14 max 9
trượt -> [9, 2, 1] tổng mới = tổng cũ - 3 + 1 = 12 | max mới phải tính lại: 9


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 `s = "abciiidef"`, `k = 3`. Liệt kê mọi cửa sổ và số nguyên âm:

| i | cửa sổ | nguyên âm | so với cửa sổ trước |
|---|---|---|---|
| 0 | abc | 1 | |
| 1 | bci | 1 | bỏ `a` (−1), thêm `i` (+1) |
| 2 | cii | 2 | bỏ `b` (0), thêm `i` (+1) |
| 3 | iii | 3 | bỏ `c` (0), thêm `i` (+1) |
| 4 | iid | 2 | bỏ `i` (−1), thêm `d` (0) |
| 5 | ide | 2 | bỏ `i` (−1), thêm `e` (+1) |
| 6 | def | 1 | bỏ `i` (−1), thêm `f` (0) |

Quan sát: hai cửa sổ kề nhau chung `k − 1` ký tự; chỉ khác **1 ký tự ra** bên trái và **1 ký tự vào** bên phải ⇒ cập nhật số đếm trong `O(1)` thay vì đếm lại `k` ký tự.

In [8]:
s, k = "abciiidef", 3
for i in range(len(s) - k + 1):
    w = s[i:i + k]
    note = "" if i == 0 else f"ra {s[i-1]!r}, vào {s[i+k-1]!r}"
    print(f"i={i} {w!r} -> {sum(c in VOWELS for c in w)} nguyên âm  {note}")

i=0 'abc' -> 1 nguyên âm  
i=1 'bci' -> 1 nguyên âm  ra 'a', vào 'i'
i=2 'cii' -> 2 nguyên âm  ra 'b', vào 'i'
i=3 'iii' -> 3 nguyên âm  ra 'c', vào 'i'
i=4 'iid' -> 2 nguyên âm  ra 'i', vào 'd'
i=5 'ide' -> 2 nguyên âm  ra 'i', vào 'e'
i=6 'def' -> 1 nguyên âm  ra 'i', vào 'f'


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi vị trí bắt đầu `i`, đếm lại nguyên âm trong `s[i:i+k]`, lấy max.

In [9]:
def max_vowels_brute(s: str, k: int) -> int:
    best = 0
    for i in range(len(s) - k + 1):
        cnt = 0
        for c in s[i:i + k]:
            if c in VOWELS:
                cnt += 1
        best = max(best, cnt)
    return best

for (s, k), want in examples:
    print(max_vowels_brute(s, k), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
2 kỳ vọng 2


**Độ phức tạp:** `(n − k + 1)` cửa sổ × `k` ký tự ⇒ `O(n·k)`; xấu nhất khi `k ≈ n/2`: `~n²/4 = 2,5·10⁹` với `n = 10^5` ⇒ TLE. Space `O(k)` cho slice.

**Nút thắt:** đếm lại `k − 1` ký tự chung giữa hai cửa sổ kề nhau.

In [10]:
rng = random.Random(0)
s_mid = "".join(rng.choice(string.ascii_lowercase) for _ in range(6000))
t = time.perf_counter()
print(max_vowels_brute(s_mid, 3000), f"brute n=6000, k=3000: {time.perf_counter()-t:.2f}s")

623 brute n=6000, k=3000: 0.30s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — prefix sum:** số nguyên âm trong `s[i:i+k]` = `P[i+k] − P[i]` ⇒ mỗi cửa sổ `O(1)` sau khi dựng `P` trong `O(n)`.

**Insight 2 — cửa sổ trượt (không cần mảng):** giữ `cnt` = số nguyên âm của cửa sổ hiện tại. Khi trượt sang phải một bước (cửa sổ mới kết thúc tại `i`):
- **nếu** `s[i]` là nguyên âm **thì** `cnt += 1` (ký tự vào),
- **nếu** `s[i−k]` là nguyên âm **thì** `cnt −= 1` (ký tự ra).

**Invariant:** sau khi xử lý chỉ số `i ≥ k − 1`, `cnt` = số nguyên âm trong `s[i−k+1 .. i]` (đúng `k` ký tự).

**Dừng sớm:** đáp án không thể vượt `k`; nếu `best == k` có thể dừng ngay.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prefix sum

**Giải thích từng đoạn:**
1. Dựng `P` độ dài `n + 1`.
2. Duyệt `i` từ `0` đến `n − k`, lấy `max(P[i + k] − P[i])`.

In [11]:
def max_vowels_prefix(s: str, k: int) -> int:
    P = [0]
    for c in s:
        P.append(P[-1] + (c in VOWELS))
    return max(P[i + k] - P[i] for i in range(len(s) - k + 1))

for (s, k), want in examples:
    print(max_vowels_prefix(s, k), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
2 kỳ vọng 2


**Code nộp LeetCode (cách 2):**

In [12]:
class SolutionPrefix:
    def maxVowels(self, s: str, k: int) -> int:
        P = [0] + list(accumulate(1 if c in "aeiou" else 0 for c in s))
        return max(P[i + k] - P[i] for i in range(len(s) - k + 1))

print(SolutionPrefix().maxVowels("abciiidef", 3))

3


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho mảng `P`. Ưu điểm: trả lời được **nhiều truy vấn** với `k` khác nhau mà chỉ dựng `P` một lần.

### 7.2 Cách 3 — Cửa sổ trượt `O(1)` bộ nhớ

**Giải thích từng đoạn:**
1. Đếm nguyên âm của cửa sổ đầu `s[0:k]` ⇒ `cnt`, `best = cnt`.
2. Với `i` từ `k` tới `n − 1`: thêm `s[i]`, bỏ `s[i − k]`, cập nhật `best`.
3. Nếu `best == k` thì dừng sớm.

In [13]:
def max_vowels_window(s: str, k: int) -> int:
    cnt = sum(c in VOWELS for c in s[:k])
    best = cnt
    for i in range(k, len(s)):
        cnt += (s[i] in VOWELS) - (s[i - k] in VOWELS)
        if cnt > best:
            best = cnt
            if best == k:          # không thể tốt hơn
                break
    return best

for (s, k), want in examples:
    print(max_vowels_window(s, k), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
2 kỳ vọng 2


**Code nộp LeetCode (cách 3)** — xem mục 9 (lời giải chính). **Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.3 Cách 4 — Cửa sổ trượt "một vòng" (không tách cửa sổ đầu)

Một khuôn dùng được cho mọi bài cửa sổ cố định: duyệt `i` từ 0, **thêm** `s[i]`; **nếu** `i ≥ k` **thì bỏ** `s[i−k]`; **nếu** `i ≥ k − 1` (cửa sổ đã đủ `k`) **thì** cập nhật đáp án. Thứ tự "thêm → bỏ → cập nhật" rất quan trọng (xem lỗi 4).

In [14]:
def max_vowels_one_loop(s: str, k: int) -> int:
    cnt = best = 0
    for i, c in enumerate(s):
        cnt += c in VOWELS                 # vào
        if i >= k:
            cnt -= s[i - k] in VOWELS      # ra
        if i >= k - 1:
            best = max(best, cnt)          # cửa sổ đủ k ký tự
    return best

class SolutionOneLoop:
    def maxVowels(self, s: str, k: int) -> int:
        vowels = set("aeiou")
        cnt = best = 0
        for i, c in enumerate(s):
            cnt += c in vowels
            if i >= k:
                cnt -= s[i - k] in vowels
            if i >= k - 1 and cnt > best:
                best = cnt
        return best

for (s, k), want in examples:
    print(max_vowels_one_loop(s, k), SolutionOneLoop().maxVowels(s, k), "kỳ vọng", want)

3 3 kỳ vọng 3
2 2 kỳ vọng 2
2 2 kỳ vọng 2


So sánh tốc độ với `n = 10^5` (bỏ brute force):

In [15]:
rng = random.Random(1456)
big = "".join(rng.choice(string.ascii_lowercase) for _ in range(10**5))
for k in (1, 1000, 50_000, 10**5):
    row = []
    for name, f in [("prefix", max_vowels_prefix), ("window", max_vowels_window), ("one_loop", max_vowels_one_loop)]:
        t = time.perf_counter()
        r = f(big, k)
        row.append(f"{name}={r} ({time.perf_counter()-t:.3f}s)")
    print(f"k={k:6d}: " + " | ".join(row))

k=     1: prefix=1 (0.015s) | window=1 (0.010s) | one_loop=1 (0.018s)
k=  1000: prefix=228 (0.025s) | window=228 (0.010s) | one_loop=228 (0.016s)
k= 50000: prefix=9696 (0.011s) | window=9696 (0.007s) | one_loop=9696 (0.014s)


k=100000: prefix=19267 (0.007s) | window=19267 (0.004s) | one_loop=19267 (0.012s)


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 3 `s = "leetcode"`, `k = 3`. Cửa sổ đầu `"lee"` có `cnt = 2`.

| i | vào s[i] | ra s[i−k] | cnt | cửa sổ | best |
|---|---|---|---|---|---|
| – | – | – | 2 | lee | 2 |
| 3 | t (0) | l (0) | 2 | eet | 2 |
| 4 | c (0) | e (−1) | 1 | etc | 2 |
| 5 | o (+1) | e (−1) | 1 | tco | 2 |
| 6 | d (0) | t (0) | 1 | cod | 2 |
| 7 | e (+1) | c (0) | 2 | ode | 2 |

In [16]:
def trace_window(s, k):
    cnt = sum(c in VOWELS for c in s[:k])
    best = cnt
    print(f"cửa sổ đầu {s[:k]!r}: cnt={cnt}, best={best}")
    for i in range(k, len(s)):
        inc, out = s[i] in VOWELS, s[i - k] in VOWELS
        cnt += inc - out
        best = max(best, cnt)
        print(f"i={i}: vào {s[i]!r}(+{int(inc)}) ra {s[i-k]!r}(-{int(out)}) -> cnt={cnt}"
              f" cửa sổ={s[i-k+1:i+1]!r} best={best}")
    return best

trace_window("leetcode", 3)

cửa sổ đầu 'lee': cnt=2, best=2
i=3: vào 't'(+0) ra 'l'(-0) -> cnt=2 cửa sổ='eet' best=2
i=4: vào 'c'(+0) ra 'e'(-1) -> cnt=1 cửa sổ='etc' best=2
i=5: vào 'o'(+1) ra 'e'(-1) -> cnt=1 cửa sổ='tco' best=2
i=6: vào 'd'(+0) ra 't'(-0) -> cnt=1 cửa sổ='cod' best=2
i=7: vào 'e'(+1) ra 'c'(-0) -> cnt=2 cửa sổ='ode' best=2


2

Trace cách 2 (prefix) trên ví dụ 1 — in mảng `P` và hiệu của từng cửa sổ:

In [17]:
def trace_prefix(s, k):
    P = [0]
    for c in s:
        P.append(P[-1] + (c in VOWELS))
    print("P =", P)
    best = 0
    for i in range(len(s) - k + 1):
        v = P[i + k] - P[i]
        best = max(best, v)
        print(f"i={i}: P[{i+k}]-P[{i}] = {P[i+k]}-{P[i]} = {v}  ({s[i:i+k]!r})  best={best}")
    return best

trace_prefix("abciiidef", 3)

P = [0, 1, 1, 1, 2, 3, 4, 4, 5, 5]
i=0: P[3]-P[0] = 1-0 = 1  ('abc')  best=1
i=1: P[4]-P[1] = 2-1 = 1  ('bci')  best=1
i=2: P[5]-P[2] = 3-1 = 2  ('cii')  best=2
i=3: P[6]-P[3] = 4-1 = 3  ('iii')  best=3
i=4: P[7]-P[4] = 4-2 = 2  ('iid')  best=3
i=5: P[8]-P[5] = 5-3 = 2  ('ide')  best=3
i=6: P[9]-P[6] = 5-4 = 1  ('def')  best=3


3

## 9. Code nộp LeetCode

Lời giải chính (cách 3 — cửa sổ trượt, dừng sớm), đúng chữ ký `maxVowels(self, s, k)`.

In [18]:
class Solution:
    def maxVowels(self, s: str, k: int) -> int:
        vowels = set("aeiou")
        cnt = sum(c in vowels for c in s[:k])
        best = cnt
        for i in range(k, len(s)):
            cnt += (s[i] in vowels) - (s[i - k] in vowels)
            if cnt > best:
                best = cnt
                if best == k:
                    break
        return best

for (s, k), want in examples:
    assert Solution().maxVowels(s, k) == want
print("Solution: 3 ví dụ đề OK")

Solution: 3 ví dụ đề OK


## 10. Vì sao thuật toán đúng

**Invariant (cách 3):** trước vòng lặp, `cnt = V(0)` với `V(j)` = số nguyên âm trong `s[j : j+k]`. Giả sử sau khi xử lý `i − 1` có `cnt = V(i − k)`. Cửa sổ mới `s[i−k+1 : i+1]` = cửa sổ cũ bỏ `s[i−k]`, thêm `s[i]`, nên `V(i−k+1) = V(i−k) − [s[i−k] là nguyên âm] + [s[i] là nguyên âm]` — đúng phép cập nhật. Vậy `cnt` lần lượt nhận mọi `V(0), V(1), …, V(n−k)`, và `best` là max của chúng.

**Dừng sớm:** `V(j) ≤ k` với mọi `j`, nên khi `best = k` thì không giá trị nào lớn hơn ⇒ dừng không làm sai.

**Cách 2:** `P[j] − P[i]` = tổng `isv[i..j−1]` = số nguyên âm trong `s[i:j]` (tính chất prefix sum); duyệt đủ `n − k + 1` vị trí.

**Cách 4:** cùng invariant; điều kiện `i ≥ k − 1` đảm bảo chỉ cập nhật khi cửa sổ có đủ `k` ký tự, và việc bỏ `s[i−k]` xảy ra **trước** khi cập nhật nên cửa sổ không bao giờ dài `k + 1` lúc được đánh giá.

## 11. Phân tích Time / Space

Gọi `n = len(s)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute | `O(n·k)` (xấu nhất `O(n²)`) | `O(k)` | slice mỗi cửa sổ |
| 2. Prefix sum | `O(n)` | `O(n)` | tốt khi nhiều truy vấn `k` |
| 3. Cửa sổ trượt | `O(n)` | `O(1)` | có dừng sớm |
| 4. Một vòng | `O(n)` | `O(1)` | khuôn chung cho cửa sổ cố định |

## 12. Bộ test

Ví dụ đề, edge case (`k = 1`, `k = n`, không có nguyên âm, toàn nguyên âm, cửa sổ tốt nhất ở **cuối** chuỗi, ở **đầu** chuỗi), đối chiếu ngẫu nhiên với brute force (seed cố định).

In [19]:
ALL = {
    "brute": max_vowels_brute,
    "prefix": max_vowels_prefix,
    "window": max_vowels_window,
    "one_loop": max_vowels_one_loop,
    "SolutionPrefix": SolutionPrefix().maxVowels,
    "SolutionOneLoop": SolutionOneLoop().maxVowels,
    "Solution": Solution().maxVowels,
}
edge = [
    (("a", 1), 1),
    (("b", 1), 0),
    (("bcdfg", 2), 0),
    (("aaaaa", 5), 5),
    (("aeiou", 5), 5),
    (("xxae", 2), 2),          # tốt nhất ở cuối
    (("aexx", 2), 2),          # tốt nhất ở đầu
    (("weallloveyou", 7), 4),
    (("tryhard", 4), 1),
    (("novowels", 1), 1),
]
for (s, k), want in examples + edge:
    for name, f in ALL.items():
        got = f(s, k)
        assert got == want, (name, s, k, got, want)
print("Ví dụ đề + edge case: OK cho", len(ALL), "cách")

Ví dụ đề + edge case: OK cho 7 cách


In [20]:
rng = random.Random(2024)
for _ in range(2000):
    n = rng.randint(1, 30)
    alphabet = rng.choice(["ab", "aeb", "aeioubcd", string.ascii_lowercase])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    k = rng.randint(1, n)
    want = max_vowels_brute(s, k)
    for name, f in ALL.items():
        assert f(s, k) == want, (name, s, k)
print("2000 test ngẫu nhiên: mọi cách khớp brute force")
for k in (1, 7, 12345, 10**5):
    assert len({f(big, k) for name, f in ALL.items() if name != "brute"}) == 1
print("Chuỗi 10^5 ký tự: các cách nhanh khớp nhau")

2000 test ngẫu nhiên: mọi cách khớp brute force


Chuỗi 10^5 ký tự: các cách nhanh khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Thiếu cửa sổ cuối: `range(n - k)` thay vì `range(n - k + 1)`

In [21]:
def wrong_range(s, k):
    return max((sum(c in VOWELS for c in s[i:i + k]) for i in range(len(s) - k)), default=0)

for s, k in [("xxae", 2), ("aeiou", 5)]:
    print(f"s={s!r}, k={k}: Sai {wrong_range(s, k)} | Đúng {Solution().maxVowels(s, k)}")

s='xxae', k=2: Sai 1 | Đúng 2
s='aeiou', k=5: Sai 0 | Đúng 5


**Sửa:** có `n − k + 1` vị trí bắt đầu ⇒ `range(len(s) - k + 1)`.

### Lỗi 2 — Quên bỏ ký tự đi ra khỏi cửa sổ

Chỉ cộng ký tự vào ⇒ `cnt` thành "số nguyên âm của cả tiền tố".

In [22]:
def wrong_no_remove(s, k):
    cnt = sum(c in VOWELS for c in s[:k])
    best = cnt
    for i in range(k, len(s)):
        cnt += s[i] in VOWELS                 # SAI: thiếu phần trừ
        best = max(best, cnt)
    return best

for s, k in [("aeiou", 2), ("leetcode", 3)]:
    print(f"s={s!r}, k={k}: Sai {wrong_no_remove(s, k)} | Đúng {Solution().maxVowels(s, k)}")

s='aeiou', k=2: Sai 5 | Đúng 2
s='leetcode', k=3: Sai 4 | Đúng 2


**Sửa:** `cnt -= s[i - k] in VOWELS`.

### Lỗi 3 — Bỏ nhầm chỉ số `s[i − k + 1]`

Ký tự rời cửa sổ khi cửa sổ mới kết thúc tại `i` là `s[i − k]` (cửa sổ mới là `s[i−k+1..i]`).

In [23]:
def wrong_index(s, k):
    cnt = sum(c in VOWELS for c in s[:k])
    best = cnt
    for i in range(k, len(s)):
        cnt += (s[i] in VOWELS) - (s[i - k + 1] in VOWELS)   # SAI
        best = max(best, cnt)
    return best

for s, k in [("abciiidef", 4), ("leetcode", 2)]:
    print(f"s={s!r}, k={k}: Sai {wrong_index(s, k)} | Đúng {Solution().maxVowels(s, k)}")

s='abciiidef', k=4: Sai 4 | Đúng 3
s='leetcode', k=2: Sai 1 | Đúng 2


**Sửa:** `s[i - k]`. Mẹo kiểm tra: với `k = 1`, ký tự ra phải là `s[i−1]`.

### Lỗi 4 — Cập nhật đáp án **trước** khi bỏ ký tự cũ (cửa sổ đang dài `k + 1`)

In [24]:
def wrong_update_order(s, k):
    cnt = best = 0
    for i, c in enumerate(s):
        cnt += c in VOWELS
        if i >= k - 1:
            best = max(best, cnt)          # SAI: lúc i >= k, cửa sổ đang có k+1 ký tự
        if i >= k:
            cnt -= s[i - k] in VOWELS
    return best

for s, k in [("aaab", 2), ("aeiou", 2)]:
    print(f"s={s!r}, k={k}: Sai {wrong_update_order(s, k)} | Đúng {Solution().maxVowels(s, k)}")

s='aaab', k=2: Sai 3 | Đúng 2
s='aeiou', k=2: Sai 3 | Đúng 2


**Sửa:** thứ tự "vào → ra → cập nhật" (cách 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chuỗi con / mảng con độ dài đúng `k`"** + đại lượng **cộng dồn được** (tổng, đếm) ⇒ **cửa sổ trượt cố định**.
- Cửa sổ cố định: chỉ cần 2 thao tác mỗi bước — **vào** `s[i]`, **ra** `s[i−k]`.
- Cần nhiều truy vấn đoạn khác nhau ⇒ **prefix sum**.
- Biết **cận trên** của đáp án (ở đây là `k`) ⇒ có thể **dừng sớm**.
- Tự kiểm tra biên: số cửa sổ là `n − k + 1`; thử `k = 1` và `k = n`.

## 15. Biến thể của bài

1. **Maximum Average Subarray I** (LeetCode 643): cửa sổ cố định trên tổng.
2. **Trả về chính chuỗi con** (vị trí đầu tiên đạt max) — bài tập 4.
3. **Cửa sổ thay đổi kích thước:** chuỗi con **dài nhất** có tối đa `t` nguyên âm — bài tập 5.
4. **Count Vowel Substrings** (LeetCode 2062): đếm chuỗi con chỉ gồm nguyên âm và chứa đủ 5 loại.
5. **Nhiều truy vấn `(l, r)`:** đếm nguyên âm trong `s[l..r]` ⇒ prefix sum (LeetCode 2559 Count Vowel Strings in Ranges).

In [25]:
def max_average_subarray(nums, k):
    # LeetCode 643 — cùng khuôn cửa sổ cố định
    cur = best = sum(nums[:k])
    for i in range(k, len(nums)):
        cur += nums[i] - nums[i - k]
        best = max(best, cur)
    return best / k

print(max_average_subarray([1, 12, -5, -6, 50, 3], 4))

def vowels_in_ranges(s, queries):
    # biến thể 5: nhiều truy vấn (l, r) bao gồm hai đầu -> prefix sum, mỗi truy vấn O(1)
    P = [0]
    for c in s:
        P.append(P[-1] + (c in VOWELS))
    return [P[r + 1] - P[l] for l, r in queries]

print(vowels_in_ranges("leetcode", [(0, 2), (3, 5), (0, 7)]))

12.75
[2, 1, 4]


**Khi nào chọn prefix sum thay vì cửa sổ trượt?** Nếu chỉ một giá trị `k` và một lần quét: cửa sổ trượt (`O(1)` bộ nhớ). Nếu có **nhiều** `k` hoặc nhiều đoạn `(l, r)` tuỳ ý: dựng prefix một lần, trả lời mỗi truy vấn `O(1)`. Ví dụ dưới so sánh: trả lời `max` cho **mọi** `k = 1..n` bằng prefix tốn `O(n²)` tổng cộng — vẫn đơn giản hơn chạy lại cửa sổ trượt cho từng `k`, và cho cùng kết quả.

In [26]:
def max_for_all_k(s):
    P = [0]
    for c in s:
        P.append(P[-1] + (c in VOWELS))
    n = len(s)
    return [max(P[i + k] - P[i] for i in range(n - k + 1)) for k in range(1, n + 1)]

res_all = max_for_all_k("leetcode")
print(res_all)
assert res_all == [max_vowels_window("leetcode", k) for k in range(1, 9)]

[1, 2, 2, 2, 3, 3, 4, 4]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Đếm lại mỗi cửa sổ | `O(n·k)` | `O(k)` | Kiểm chứng |
| 2. Prefix sum | `P[i+k] − P[i]` | `O(n)` | `O(n)` | Nhiều truy vấn |
| 3. Cửa sổ trượt | Vào +, ra −, dừng sớm | `O(n)` | `O(1)` | Lời giải chuẩn |
| 4. Một vòng | Vào → ra → cập nhật | `O(n)` | `O(1)` | Khuôn chung, ít nhánh đặc biệt |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `maxVowels("tryhard", 4)` = ?

**Bài 2 (tính tay):** liệt kê số nguyên âm của **mọi** cửa sổ độ dài 3 trong `"leetcode"`.

**Bài 3 (điền TODO):** hoàn thiện `max_vowels_todo` (cửa sổ trượt).

**Bài 4 (tự viết lại):** viết `best_substring(s, k)` trả về **chuỗi con** độ dài `k` có nhiều nguyên âm nhất; nếu nhiều, lấy chuỗi xuất hiện **sớm nhất**.

**Bài 5 (mini challenge):** viết `longest_at_most_t_vowels(s, t)` trả về độ dài chuỗi con **dài nhất** chứa **không quá `t`** nguyên âm (cửa sổ thay đổi kích thước, `O(n)`).

In [27]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def max_vowels_todo(s, k):
    cnt = 0
    # TODO: đếm nguyên âm trong k ký tự đầu vào cnt
    best = cnt
    for i in range(k, len(s)):
        # TODO: cộng nếu s[i] là nguyên âm, trừ nếu s[i - k] là nguyên âm
        # TODO: cập nhật best
        pass
    return best

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Nguyên âm duy nhất là `a`. (2) Cửa sổ dài 4 có chứa `a` không? (3) `"hard"` có 1.

**Bài 2:** (1) Có `8 − 3 + 1 = 6` cửa sổ. (2) Dùng trace mục 8. (3) `[2, 2, 1, 1, 1, 2]`.

**Bài 3:** (1) `cnt = sum(c in VOWELS for c in s[:k])`. (2) `cnt += (s[i] in VOWELS) - (s[i-k] in VOWELS)`. (3) `best = max(best, cnt)`.

**Bài 4:** (1) Lưu vị trí bắt đầu `start` khi `cnt > best` (strict ⇒ giữ vị trí sớm nhất). (2) Cửa sổ kết thúc tại `i` bắt đầu tại `i − k + 1`. (3) Trả về `s[start:start+k]`.

**Bài 5:** (1) Hai con trỏ `l, r`; thêm `s[r]`. (2) Trong khi số nguyên âm > `t` thì bỏ `s[l]`, `l += 1`. (3) Cập nhật `r − l + 1`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
ans1 = Solution().maxVowels("tryhard", 4)
print("Bài 1:", ans1)
assert ans1 == 1

# Bài 2
ans2 = [sum(c in VOWELS for c in "leetcode"[i:i + 3]) for i in range(6)]
print("Bài 2:", ans2)
assert ans2 == [2, 2, 1, 1, 1, 2]

# Bài 3
def max_vowels_todo_sol(s, k):
    cnt = sum(c in VOWELS for c in s[:k])
    best = cnt
    for i in range(k, len(s)):
        cnt += (s[i] in VOWELS) - (s[i - k] in VOWELS)
        best = max(best, cnt)
    return best

for (s, k), want in examples + edge:
    assert max_vowels_todo_sol(s, k) == want
print("Bài 3: OK")

Bài 1: 1
Bài 2: [2, 2, 1, 1, 1, 2]
Bài 3: OK


In [29]:
# Bài 4
def best_substring(s, k):
    cnt = sum(c in VOWELS for c in s[:k])
    best, start = cnt, 0
    for i in range(k, len(s)):
        cnt += (s[i] in VOWELS) - (s[i - k] in VOWELS)
        if cnt > best:
            best, start = cnt, i - k + 1
    return s[start:start + k]

def best_substring_brute(s, k):
    subs = [s[i:i + k] for i in range(len(s) - k + 1)]
    m = max(sum(c in VOWELS for c in w) for w in subs)
    return next(w for w in subs if sum(c in VOWELS for c in w) == m)

# Bài 5
def longest_at_most_t_vowels(s, t):
    l = cnt = best = 0
    for r, c in enumerate(s):
        cnt += c in VOWELS
        while cnt > t:
            cnt -= s[l] in VOWELS
            l += 1
        best = max(best, r - l + 1)
    return best

def longest_brute(s, t):
    n = len(s)
    return max((j - i for i in range(n) for j in range(i, n + 1)
                if sum(c in VOWELS for c in s[i:j]) <= t), default=0)

assert best_substring("abciiidef", 3) == "iii"
assert longest_at_most_t_vowels("leetcode", 1) == 4      # "tcod"
rng = random.Random(19)
for _ in range(500):
    s = "".join(rng.choice("aeibcd") for _ in range(rng.randint(1, 15)))
    k = rng.randint(1, len(s))
    t = rng.randint(0, 4)
    assert best_substring(s, k) == best_substring_brute(s, k)
    assert longest_at_most_t_vowels(s, t) == longest_brute(s, t)
print("Bài 4:", best_substring("leetcode", 3), "| Bài 5:", longest_at_most_t_vowels("leetcode", 1), "| OK (500 test)")

Bài 4: lee | Bài 5: 4 | OK (500 test)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template cửa sổ trượt cố định:**

```python
cnt = sum(is_good(c) for c in s[:k])
best = cnt
for i in range(k, len(s)):
    cnt += is_good(s[i]) - is_good(s[i - k])   # vào - ra
    best = max(best, cnt)
```

**Template một vòng:**

```python
for i, c in enumerate(s):
    cnt += is_good(c)                      # vào
    if i >= k: cnt -= is_good(s[i - k])    # ra
    if i >= k - 1: best = max(best, cnt)   # cập nhật
```

**Checklist:**
- [ ] Số cửa sổ `n − k + 1` (không thiếu cửa sổ cuối).
- [ ] Mỗi bước có cả **vào** `s[i]` và **ra** `s[i − k]`.
- [ ] Cập nhật đáp án khi cửa sổ có **đúng `k`** ký tự.
- [ ] Đã thử `k = 1`, `k = n`, không có nguyên âm, toàn nguyên âm.